# Quant Safety Audit — Phase 0: Setup, Verification & Quantization

This notebook does three things, in this order, on purpose:

1. **Verify the model source is legitimate** before we trust it (Alibaba's Qwen
   naming has gotten confusing — `Qwen3.6` is real, but a lot of HF repos with
   "Qwen3.6" in the name are *not* published by the Qwen org. We check this
   programmatically rather than trusting the repo name.)
2. **Confirm every benchmark dataset actually downloads and looks correct**
   before we generate a single model response.
3. **Quantize the same model with 6 publicly available methods** and save each
   result to Google Drive so it survives when this Colab session ends.

Run top to bottom the first time. After that, cells are independent per method —
you don't need to re-run quantization methods you've already produced.

**Runtime:** Runtime → Change runtime type → GPU (T4 is enough for a 3B model).


## 0. Setup

In [ ]:
# Core libraries. gptqmodel / autoawq / hqq / auto-round are the quantization
# backends; the rest are standard HF tooling for loading models + datasets.
!pip install -q -U transformers accelerate huggingface_hub datasets
!pip install -q -U bitsandbytes
!pip install -q -U hqq
!pip install -q -U autoawq
!pip install -q -U gptqmodel
!pip install -q -U auto-round
print("Install step done. If any line above errored, re-run just that pip install alone to see the real error.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Everything we produce or download gets saved under this Drive folder so it
# survives a Colab disconnect. Change this path if you want a different location.
OUTPUT_DIR = "/content/drive/MyDrive/quant-safety-audit"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/models", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/benchmarks", exist_ok=True)
print("Output directory:", OUTPUT_DIR)


## 1. Config — pick the model here

**Recommendation: `Qwen/Qwen2.5-3B-Instruct`.** Not the newest Qwen release —
that's deliberate. Verified checkpoint availability beats novelty for this
project:

- `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` — **official** Qwen GPTQ checkpoint exists.
- `bartowski/Qwen2.5-3B-Instruct-GGUF` — extensive, well-maintained GGUF k-quants exist.
- bitsandbytes and HQQ need no pre-quantized checkpoint at all — we produce those ourselves.
- AWQ: check for an official/community checkpoint at run time (cell below); self-quantize if none exists.

Two models to actively avoid for this project, and why:
- `Qwen/Qwen3.5-4B` — confirmed **multimodal** (image-text-to-text), not a plain
  text model. Adds VLM-specific complexity to every quantization tool below,
  several of which have shakier VLM support. Skip it for this project.
- `Qwen3.6` family — the only open-weight checkpoints are `Qwen3.6-27B` (dense)
  and `Qwen3.6-35B-A3B` (MoE); both are too large to quantize down to something
  that fits comfortably on a small consumer GPU / this project's compute budget.
  `Qwen3.6-Plus` / `Qwen3.6-Max-Preview` are hosted API-only, not open weights.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Swap-in candidates for later, once this pipeline is proven on Qwen2.5-3B.
# Uncomment ONE to change the target model — everything below reads MODEL_ID,
# so nothing else needs to change.
# MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"     # newer generation, text-only, dense
# MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct" # different family entirely, for cross-family replication later

print("Target model:", MODEL_ID)


## 2. Phase 0a — Verify the model source before trusting it

This is the check that catches "someone re-uploaded a model under a name that
looks official but isn't." We hit the HF API directly rather than eyeballing
the repo name.


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
info = api.model_info(MODEL_ID)

EXPECTED_OFFICIAL_ORGS = {"Qwen", "meta-llama", "mistralai", "google", "microsoft"}
author = MODEL_ID.split("/")[0]

print(f"Model ID:        {MODEL_ID}")
print(f"Author/org:      {author}")
print(f"Official org?    {'YES' if author in EXPECTED_OFFICIAL_ORGS else 'NO -- verify manually before proceeding'}")
print(f"License:         {info.card_data.get('license') if info.card_data else 'unknown -- check model card manually'}")
print(f"Tags:            {info.tags}")
print(f"Downloads:       {info.downloads}")
print(f"Last modified:   {info.last_modified}")

# Flag if this looks like a multimodal/VLM checkpoint rather than plain text --
# several quantization tools below assume a text-only causal LM.
multimodal_flags = [t for t in info.tags if any(k in t.lower() for k in ["image-text", "vision", "vl", "multimodal"])]
if multimodal_flags:
    print(f"\nWARNING: this model looks multimodal (tags: {multimodal_flags}).")
    print("Several quantization tools below assume text-only causal LM input. Reconsider before proceeding.")
else:
    print("\nNo multimodal tags detected -- looks like a plain text model. Good.")


## 3. Phase 0b — Confirm the benchmark datasets actually download and look right

Same principle as above: don't trust a dataset name, look at what actually
comes back. Each cell below downloads a small piece and prints a couple of
raw examples so you can eyeball them before building anything on top.


In [ ]:
# --- HarmBench (under-refusal / harmful-compliance test set) ---
# Public GitHub repo, no auth needed.
!git clone -q https://github.com/centerforaisafety/HarmBench.git {OUTPUT_DIR}/benchmarks/harmbench 2>/dev/null || echo "Already cloned or clone failed -- check manually if this is unexpected."

import glob
harmbench_csvs = glob.glob(f"{OUTPUT_DIR}/benchmarks/harmbench/**/*.csv", recursive=True)
print(f"Found {len(harmbench_csvs)} CSV files under HarmBench.")
print("First few:", harmbench_csvs[:5])


In [ ]:
# --- OR-Bench (over-refusal test set) ---
from datasets import load_dataset

or_bench = load_dataset("bench-llm/or-bench", "or-bench-hard-1k")
print(or_bench)
print("\nSample row:")
print(or_bench["train"][0])


In [ ]:
# --- ToxiGen (toxicity test set, ships with its own classifier) ---
toxigen = load_dataset("toxigen/toxigen-data", "annotated")
print(toxigen)
print("\nSample row:")
print(toxigen["test"][0])


In [ ]:
# --- BIPIA (prompt-injection resistance) -- email subtask only ---
# The email subtask is self-contained (no extra licensed corpus needed to
# reconstruct context data, unlike the Web QA / Summarization subtasks).
!git clone -q https://github.com/microsoft/BIPIA.git {OUTPUT_DIR}/benchmarks/bipia 2>/dev/null || echo "Already cloned or clone failed -- check manually if this is unexpected."

import os
email_dir = f"{OUTPUT_DIR}/benchmarks/bipia/benchmark/email"
if os.path.exists(email_dir):
    print("Email subtask files:", os.listdir(email_dir))
else:
    print("Email subtask directory not found at expected path -- check the repo structure manually, it may have moved.")


In [ ]:
# --- sycophancy-eval (pushback resistance) ---
sycophancy = load_dataset("meg-tong/sycophancy-eval")
print(sycophancy)
print("\nSample 'are_you_sure' row (this is the pushback-resistance file we'll use):")
print(sycophancy["are_you_sure"][0])


In [ ]:
# --- Manifest: confirm everything above actually resolved before moving on ---
print("Phase 0b check:")
print(f"  HarmBench CSVs found:        {len(harmbench_csvs)}")
print(f"  OR-Bench rows:               {len(or_bench['train'])}")
print(f"  ToxiGen rows:                {len(toxigen['test'])}")
print(f"  BIPIA email dir present:     {os.path.exists(email_dir)}")
print(f"  sycophancy-eval loaded:      {'are_you_sure' in sycophancy}")
print("\nIf every line above looks right, Phase 0 is done -- proceed to quantization.")
print("If anything is missing or looks wrong, STOP and fix it here before generating any model output.")


## 4. Quantization — six methods, same model

Same model (`MODEL_ID`), six ways of compressing it. Where an official or
well-maintained community checkpoint already exists, we download it directly
rather than re-quantizing ourselves — that's actually more representative of
what a real user deploying this model would get, and it saves compute.
Where no such checkpoint exists, we quantize it ourselves in-notebook.

Each method's cell is independent — run only the ones you need.


### 4.1 bitsandbytes (NF4 and INT8) — load-time quantization, fastest to test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

for bnb_variant, bnb_config in [
    ("nf4", BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)),
    ("int8", BitsAndBytesConfig(load_in_8bit=True)),
]:
    save_path = f"{OUTPUT_DIR}/models/bnb-{bnb_variant}"
    if os.path.exists(save_path):
        print(f"[bnb-{bnb_variant}] already exists at {save_path}, skipping.")
        continue
    print(f"[bnb-{bnb_variant}] loading + quantizing...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"[bnb-{bnb_variant}] saved to {save_path}")
    del model
    torch.cuda.empty_cache()


### 4.2 HQQ — data-free, no calibration set needed, fastest method to produce

In [ ]:
from hqq.engine.hf import HQQModelForCausalLM
from hqq.core.quantize import BaseQuantizeConfig

save_path = f"{OUTPUT_DIR}/models/hqq-4bit"
if os.path.exists(save_path):
    print(f"[hqq] already exists at {save_path}, skipping.")
else:
    print("[hqq] loading + quantizing (data-free, should be fast)...")
    hqq_config = BaseQuantizeConfig(nbits=4, group_size=64)
    model = HQQModelForCausalLM.from_pretrained(MODEL_ID)
    model.quantize_model(quant_config=hqq_config)
    model.save_quantized(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"[hqq] saved to {save_path}")


### 4.3 GGUF — download the community k-quant build

We download rather than convert ourselves for now (bartowski's builds are
widely used and well-vetted). If you switch `MODEL_ID` to a model with no
pre-built GGUF, you'll need to build llama.cpp and convert/quantize manually --
flagging that as a TODO for a future model, not needed for Qwen2.5-3B.


In [ ]:
from huggingface_hub import hf_hub_download

GGUF_REPO = "bartowski/Qwen2.5-3B-Instruct-GGUF"
GGUF_FILES = [
    "Qwen2.5-3B-Instruct-Q4_K_M.gguf",  # primary comparison point -- roughly matches the ~4-bit level used by the other methods
    "Qwen2.5-3B-Instruct-Q8_0.gguf",    # optional higher-precision reference within the same method
]

save_dir = f"{OUTPUT_DIR}/models/gguf"
os.makedirs(save_dir, exist_ok=True)
for fname in GGUF_FILES:
    dest = f"{save_dir}/{fname}"
    if os.path.exists(dest):
        print(f"[gguf] {fname} already downloaded, skipping.")
        continue
    print(f"[gguf] downloading {fname} ...")
    path = hf_hub_download(repo_id=GGUF_REPO, filename=fname, local_dir=save_dir)
    print(f"[gguf] saved to {path}")


### 4.4 GPTQ — official Qwen checkpoint first, self-quantize only as a fallback


In [ ]:
save_path = f"{OUTPUT_DIR}/models/gptq-int4"
OFFICIAL_GPTQ_ID = "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4"

if os.path.exists(save_path):
    print(f"[gptq] already exists at {save_path}, skipping.")
else:
    try:
        print(f"[gptq] checking for official checkpoint: {OFFICIAL_GPTQ_ID}")
        api.model_info(OFFICIAL_GPTQ_ID)  # raises if it doesn't exist
        print("[gptq] official checkpoint found -- downloading it directly rather than self-quantizing.")
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=OFFICIAL_GPTQ_ID, local_dir=save_path)
        print(f"[gptq] saved to {save_path}")
    except Exception as e:
        print(f"[gptq] no official checkpoint found ({e}) -- self-quantizing with GPTQModel instead.")
        print("[gptq] this needs a calibration set -- using a small slice of wikitext as a generic calibration corpus.")
        from gptqmodel import GPTQModel, QuantizeConfig
        from datasets import load_dataset as _ld

        calibration_data = _ld("wikitext", "wikitext-2-raw-v1", split="train")["text"][:256]
        calibration_data = [t for t in calibration_data if len(t.strip()) > 0][:128]

        quant_config = QuantizeConfig(bits=4, group_size=128)
        model = GPTQModel.load(MODEL_ID, quant_config)
        model.quantize(calibration_data)
        model.save(save_path)
        print(f"[gptq] self-quantized and saved to {save_path}")


### 4.5 AWQ — official/community checkpoint first, self-quantize only as a fallback


In [ ]:
save_path = f"{OUTPUT_DIR}/models/awq-int4"
CANDIDATE_AWQ_IDS = [
    "Qwen/Qwen2.5-3B-Instruct-AWQ",   # check official org first
]

if os.path.exists(save_path):
    print(f"[awq] already exists at {save_path}, skipping.")
else:
    found = None
    for candidate in CANDIDATE_AWQ_IDS:
        try:
            api.model_info(candidate)
            found = candidate
            break
        except Exception:
            continue

    if found:
        print(f"[awq] found existing checkpoint: {found} -- downloading directly.")
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=found, local_dir=save_path)
        print(f"[awq] saved to {save_path}")
    else:
        print("[awq] no ready-made checkpoint found in the candidate list -- self-quantizing with AutoAWQ.")
        print("[awq] this needs a calibration set and will take longer than the methods above -- expect real GPU time here.")
        from awq import AutoAWQForCausalLM

        model = AutoAWQForCausalLM.from_pretrained(MODEL_ID)
        awq_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
        model.quantize(awq_tokenizer, quant_config=quant_config)
        model.save_quantized(save_path)
        awq_tokenizer.save_pretrained(save_path)
        print(f"[awq] self-quantized and saved to {save_path}")


### 4.6 AutoRound — Intel's current leaderboard-topping PTQ method (2026)

Newer than the classic four above and worth including precisely because it's
recent: AutoRound has been reported to outperform GPTQ/AWQ/GGUF on low-bit
accuracy leaderboards, especially for small-to-medium models like this one.
No widely available pre-quantized checkpoint for a 3B Qwen model yet, so we
quantize it ourselves.


In [ ]:
save_path = f"{OUTPUT_DIR}/models/autoround-int4"

if os.path.exists(save_path):
    print(f"[autoround] already exists at {save_path}, skipping.")
else:
    print("[autoround] self-quantizing with auto-round -- this is calibration-based, expect real GPU time.")
    from auto_round import AutoRound

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    ar_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    autoround = AutoRound(model, ar_tokenizer, bits=4, group_size=128)
    autoround.quantize()
    autoround.save_quantized(save_path, format="auto_round")
    print(f"[autoround] saved to {save_path}")


## 5. Manifest — confirm what actually got produced

In [ ]:
import os

print(f"{'method':<20}{'path':<55}{'size (GB)'}")
print("-" * 90)
for method_dir in sorted(os.listdir(f"{OUTPUT_DIR}/models")):
    full_path = f"{OUTPUT_DIR}/models/{method_dir}"
    total_bytes = 0
    for root, _, files in os.walk(full_path):
        for f in files:
            total_bytes += os.path.getsize(os.path.join(root, f))
    print(f"{method_dir:<20}{full_path:<55}{total_bytes / 1e9:.2f}")

print("\nEverything above is saved under your Google Drive at:", f"{OUTPUT_DIR}/models")
print("Download locally with: from google.colab import files -- or just sync the Drive folder.")
